##installed all libraries


In [ ]:
!pip install sentence-transformers faiss-cpu groq pypdf python-docx tqdm -q
print(" All libraries installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.7 MB/s eta 0:00:00
 All libraries installed!


#API key

In [6]:

from google.colab import userdata
api_key = userdata.get("GROQ_API_KEY")

In [ ]:
import os

os.makedirs("/content/knowledge_base", exist_ok=True)

## code to prepare knowlegde base

In [ ]:
import re
from pathlib import Path
from typing import List, Dict


# =========================
# 1. TEXT EXTRACTION
# =========================
def extract_text(file_path: str) -> str:
    """Extract text from .txt / .pdf / .docx files"""
    path = Path(file_path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    file_name = path.name.lower()

    try:
        if file_name.endswith((".txt", ".md")):
         return path.read_text(encoding="utf-8", errors="ignore")

        elif file_name.endswith(".pdf"):
         from pypdf import PdfReader
         reader = PdfReader(str(path))
         return "\n".join(page.extract_text() or "" for page in reader.pages)

        elif file_name.endswith(".docx"):
         import docx
         doc = docx.Document(str(path))
         return "\n".join(p.text for p in doc.paragraphs)

        else:
         raise ValueError(f"Unsupported format: {file_name}")

    except Exception as e:
        print(f"❌ Error reading {file_path}: {e}")
        return ""


# =========================
# 2. CLEAN TEXT
# =========================
def clean_text(text: str) -> str:
    """Basic text cleaning"""
    text = re.sub(r"\s+", " ", text)      # remove extra spaces
    text = re.sub(r"\n+", "\n", text)     # normalize newlines
    return text.strip()


# =========================
# 3. CHUNKING (RAG OPTIMIZED)
# =========================
def chunk_text(text: str, source: str,
               chunk_size: int = 300,
               overlap: int = 50) -> List[Dict]:
    """
    Split text into overlapping chunks
    Returns list of dict with metadata
    """
    text = clean_text(text)
    words = text.split()

    chunks = []
    start = 0
    chunk_id = 0

    while start < len(words):
        end = start + chunk_size
        chunk_words = words[start:end]

        if not chunk_words:
            break

        chunk = " ".join(chunk_words)

        chunks.append({
            "text": chunk,
            "source": source,
            "chunk_id": chunk_id,
            "length": len(chunk_words)
        })

        chunk_id += 1
        start += (chunk_size - overlap)

    return chunks


# =========================
# 4. LOAD MULTIPLE FILES
# =========================
def load_knowledge_base(folder_path: str) -> List[Dict]:
    """
    Load all supported files from a folder
    and convert into chunks
    """
    folder = Path(folder_path)

    if not folder.exists():
        raise FileNotFoundError("Folder not found!")

    all_chunks = []

    for file in folder.glob("*"):
        if file.suffix.lower() in [".txt", ".pdf", ".docx", ".md"]:
            print(f"📄 Processing: {file.name}")

            text = extract_text(str(file))

            if text.strip():
                chunks = chunk_text(text, source=file.name)
                all_chunks.extend(chunks)

    print(f"✅ Total chunks created: {len(all_chunks)}")
    return all_chunks


# =========================
# TEST RUN
# =========================
if __name__ == "__main__":
    data = load_knowledge_base("knowledge_base")

    print("\n🔹 Sample Chunk:")
    print(data[0])

📄 Processing: Task Validation Rules.txt
📄 Processing: Meeting Scheduling Process.txt
📄 Processing: Intern Performance Evaluation.txt
📄 Processing: Internship Guidelines.txt
📄 Processing: Attendance Policy.txt
✅ Total chunks created: 5

🔹 Sample Chunk:
{'text': "Task Validation Rules To ensure high-quality output and maintain project standards, all completed tasks must go through a validation process. 1. Task Submission - Completion: A task is considered complete only when it meets all the acceptance criteria defined in the task description. - Documentation: Provide necessary documentation, code comments, or brief reports explaining how you completed the task. - Platform Update: Move the task to the 'In Review' or indicating status on the UptoSkills project management board. 2. Code and Technical Reviews (If Applicable) - Pull Requests: For software development tasks, create a pull request (PR) for peer and mentor review. - Testing: Ensure all code passes unit tests and linting checks b

In [ ]:
import os
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# =========================
# STEP 1: Load + Chunk
# =========================
print("📂 Loading Knowledge Base...")

folder = "knowledge_base"
files = [f for f in os.listdir(folder) if f.endswith((".txt", ".pdf", ".docx"))]

all_chunks = []
for f in files:
    fp = f"{folder}/{f}"
    text = extract_text(fp)
    all_chunks += chunk_text(text, source=f)

print(f"✅ Total chunks: {len(all_chunks)}")

# =========================
# STEP 2: Embeddings
# =========================
print("\n⚡ Generating Embeddings...")

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c["text"] for c in all_chunks]

embeddings = model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True
).astype("float32")

# =========================
# STEP 3: FAISS Index
# =========================
print("\n🔎 Building FAISS Index...")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print(f"🎉 Ready! {index.ntotal} vectors stored")

📂 Loading Knowledge Base...
✅ Total chunks: 5

⚡ Generating Embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


🔎 Building FAISS Index...
🎉 Ready! 5 vectors stored


## pipeline ready

In [ ]:
from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

SYSTEM_PROMPT = """You are the SkillNova Knowledge Base Assistant.

Rules:
- Answer ONLY from given context.
- If not found, say: "I could not find that in the knowledge base."
- Be concise and professional.
- Use bullet points if needed.
"""

# =========================
# RETRIEVAL
# =========================
def retrieve(question: str, top_k: int = 4):
    q_vec = model.encode([question], normalize_embeddings=True).astype("float32")
    scores, idxs = index.search(q_vec, top_k)

    return [
        {**all_chunks[i], "score": float(s)}
        for s, i in zip(scores[0], idxs[0]) if i >= 0
    ]


# =========================
# CHAT (RAG PIPELINE)
# =========================
def chat(question: str):
    retrieved = retrieve(question)

    if not retrieved:
        return {"answer": "I could not find that in the knowledge base.", "sources": []}

    # Build context (limit size for token efficiency)
    context = "\n\n".join(
        f"[{c['source']}]\n{c['text'][:300]}"
        for c in retrieved
    )

    prompt = f"CONTEXT:\n{context}\n\nQUESTION: {question}\n\nAnswer:"

    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        temperature=0.2,
        max_tokens=400,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ]
    )

    return {
        "answer": response.choices[0].message.content.strip(),
        "sources": list({c["source"] for c in retrieved})
    }

## test questions

In [ ]:
test_questions = [
    "What are the internship guidelines?",
    "How do I submit weekly reports?",
    "When is the meeting scheduled?",
    "How is attendance calculated?",
    "How are tasks validated?",
    "How to earn internship certificate without delay?",
    "How to increase my rating?",
    "How is my performance evaluated?",
]

print("\n🧪 Running Chatbot Tests...\n" + "="*60)

for i, q in enumerate(test_questions, 1):
    result = chat(q)

    print(f"\n[{i}] ❓ {q}")
    print(f"🤖 {result['answer']}")

    if result["sources"]:
        print(f"📄 Sources: {', '.join(result['sources'])}")
    else:
        print("📄 Sources: None")

print("\n✅ Testing Completed!")


🧪 Running Chatbot Tests...

[1] ❓ What are the internship guidelines?
🤖 The internship guidelines are outlined in the "Internship Guidelines.txt" file. Key points include:

* Maintaining professional conduct and adhering to a code of conduct
* Ensuring consistent attendance and active participation
* Following the meeting scheduling process for effective communication

For a detailed view, the guidelines are as follows:

1. Professional Conduct - Code of Conduct: 
All interns are expected to maintain professional conduct and adhere to a code of conduct.

2. Attendance Policy:
- Consistent attendance and active participation are essential for maximizing the value of your internship at UptoSkills.
- Interns are expected to adhere to the agreed-upon standard working hours for their specific role and time zone.
- Flexibility is also expected in certain situations.

3. Meeting Scheduling Process:
- Effective communication is crucial for the success of your internship.
- Use the designated 

##installing gradio

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr

# =========================
# CHAT FUNCTION
# =========================
def ask_chatbot(question):
    question = question.strip()

    if not question:
        return "⚠️ Please enter a valid question.", ""

    try:
        result = chat(question)
        answer  = result.get("answer", "No response generated.")
        sources = ", ".join(result.get("sources", [])) or "No sources found"

        return answer, sources

    except Exception as e:
        return f"❌ Error: {str(e)}", ""


# =========================
# UI DESIGN
# =========================
with gr.Blocks(title="SkillNova Chatbot", theme=gr.themes.Soft()) as app:

    gr.Markdown("""
    # 🤖 SkillNova Knowledge Base Chatbot
    Ask questions related to UptoSkills internship policies
    """)

    with gr.Row():
        with gr.Column(scale=3):
            question_box = gr.Textbox(
                label="💬 Your Question",
                placeholder="e.g. How do I submit weekly reports?",
                lines=2
            )

            ask_btn = gr.Button("🚀 Ask", variant="primary")

        with gr.Column(scale=1):
            source_box = gr.Textbox(
                label="📄 Sources",
                lines=3,
                interactive=False
            )

    answer_box = gr.Textbox(
        label="🤖 Answer",
        lines=8,
        interactive=False
    )

    # =========================
    # EXAMPLES
    # =========================
    gr.Examples(
        examples=[
            "What are the internship guidelines?",
            "How do I submit weekly reports?",
            "How is attendance calculated?",
            "How are tasks validated?",
            "How to earn internship certificate without delay?",
            "How to increase my rating?",
            "How is my performance evaluated?",
            "How do I schedule a meeting?",
        ],
        inputs=question_box
    )

    # =========================
    # BUTTON ACTIONS
    # =========================
    ask_btn.click(
        fn=ask_chatbot,
        inputs=question_box,
        outputs=[answer_box, source_box]
    )

    question_box.submit(
        fn=ask_chatbot,
        inputs=question_box,
        outputs=[answer_box, source_box]
    )

# =========================
# LAUNCH APP
# =========================
app.launch(share=True)

/tmp/ipykernel_5377/1878748807.py:26: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SkillNova Chatbot", theme=gr.themes.Soft()) as app:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://997d1b089914de152a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
